# Detecção de Áudio Falso — ASVspoof 2021 LA
PCS5917 IA Adversarial (Poli/USP) — Leonardo Kawakami Hissanaga, NUSP 11374864.

**Substituição de fonte registrada:** o Kaggle exige login; usamos o ASVspoof 2021 LA na república pública `MoaazTalab/ASVspoof_2021_LA_Balanced_Normalized` (Hugging Face). Pipeline: amostra balanceada 1.200 áudios → features (MFCC/espectrais/bandas/chroma via librosa) → RandomForest × GradientBoosting × SVM-RBF com CV 5× e sensibilidade a subconjuntos de features.

In [ ]:
!pip -q install datasets librosa scikit-learn soundfile matplotlib
import os, io, wave, json, random, numpy as np, pandas as pd
from datasets import load_dataset
from collections import Counter

random.seed(42)
TARGET = 300  # por banda por split

ds = load_dataset("MoaazTalab/ASVspoof_2021_LA_Balanced_Normalized")
rows = []
for split in ("train", "validation"):
    buckets = {}
    d = ds[split]
    idx = list(range(len(d))); random.shuffle(idx)
    for i in idx:
        b = str(int(d[i]["label"]))
        if len(buckets.get(b, [])) >= TARGET: 
            if all(len(v) >= TARGET for v in buckets.values()) and len(buckets) >= 2: break
            continue
        buckets.setdefault(b, []).append(d[i]["audio"])
        if len(buckets) >= 2 and all(len(v) >= TARGET for v in buckets.values()): break
    take = min(TARGET, min(len(v) for v in buckets.values()))
    for band, items in buckets.items():
        for au in items[:take]:
            rows.append({"split": split, "band": band, "audio": au})
    print(split, {k: len(v) for k, v in buckets.items()})

os.makedirs("/content/amostras", exist_ok=True)
index = []
for j, row in enumerate(rows):
    au = row["audio"]
    x = np.asarray(au["array"], dtype="float32"); sr = au["sampling_rate"]
    key = f"{row['split']}_{row['band']}_{j:04d}"
    pcm = (np.clip(x, -1, 1) * 32767).astype("<i2")
    with wave.open(f"/content/amostras/{key}.wav", "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes(pcm.tobytes())
    index.append({"id": key, "split": row["split"], "label": int(row["band"])})
    if j and j % 200 == 0: print(j, "gravados")
print("total:", len(index), Counter(r["label"] for r in index))

## Features (estilo PyAudioAnalysis, via librosa)
MFCC 1–13 (média+std), ZCR, RMS, centroide/bandwidth/rolloff, *spectral flatness*, energia relativa em 4 bandas (0–500 Hz / 0,5–2 / 2–4 / 4–8 kHz), chroma (12), duração. Hipótese: artefatos de síntese reduzem a energia acima de 4 kHz e alteram a flatness.

In [ ]:
import librosa
AMOSTRAS = "/content/amostras"

def read_wav(path):
    with wave.open(path, "rb") as w:
        sr = w.getframerate(); x = np.frombuffer(w.readframes(w.getnframes()), dtype="<i2").astype("float32") / 32767.0
    return x, sr

def features(x, sr):
    f = {}
    if len(x) < sr // 10: x = np.pad(x, (0, sr - len(x)))
    mfcc = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13)
    for i in range(13):
        f[f"mfcc{i:02d}_mean"] = mfcc[i].mean(); f[f"mfcc{i:02d}_std"] = mfcc[i].std()
    f["zcr_mean"] = float(np.mean(np.abs(np.diff(np.sign(x)))))
    rms = librosa.feature.rms(y=x)[0]; f["rms_mean"], f["rms_std"] = rms.mean(), rms.std()
    cent = librosa.feature.spectral_centroid(y=x, sr=sr)[0]
    bw = librosa.feature.spectral_bandwidth(y=x, sr=sr)[0]
    ro = librosa.feature.spectral_rolloff(y=x, sr=sr)[0]
    fl = librosa.feature.spectral_flatness(y=x)[0]
    f.update({"centroid_mean": cent.mean(), "centroid_std": cent.std(), "bandwidth_mean": bw.mean(),
              "rolloff_mean": ro.mean(), "flatness_mean": fl.mean(), "flatness_std": fl.std()})
    S = np.abs(librosa.stft(x)) ** 2; freqs = librosa.fft_frequencies(sr=sr); tot = S.sum() + 1e-12
    for j, (lo, hi) in enumerate([(0, 500), (500, 2000), (2000, 4000), (4000, 8000)]):
        m = (freqs >= lo) & (freqs < hi); f[f"band{j}"] = float(S[m].sum() / tot)
    ch = librosa.feature.chroma_cqt(y=x, sr=sr)
    for i in range(12): f[f"chroma{i:02d}"] = ch[i].mean()
    f["duracao"] = len(x) / sr
    return f

rows_f = []
for r in index:
    x, sr = read_wav(f"{AMOSTRAS}/{r['id']}.wav")
    g = features(x, sr); g.update({"id": r["id"], "split": r["split"], "label": r["label"]})
    rows_f.append(g)
df = pd.DataFrame(rows_f)
print(df.shape)
df.head()

## Exploração: onde está o sinal?
As bandas de alta frequência separam as classes — o teto espectral é mais baixo nas falsas:

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for lab, c in [(0, "tab:red"), (1, "tab:blue")]:
    ax[0].hist(df[df.label == lab]["band3"], bins=30, alpha=.6, density=True, color=c, label="fake" if lab == 0 else "real")
ax[0].set_title("Energia relativa 4–8 kHz"); ax[0].legend()
ax[1].hist([df[df.label == 0]["flatness_mean"], df[df.label == 1]["flatness_mean"]], bins=30, density=True, label=["fake", "real"])
ax[1].set_title("Spectral flatness"); ax[1].legend(); plt.tight_layout(); plt.show()

## Classificadores + CV 5× estratificada
RandomForest × GradientBoosting × SVM-RBF (com scaling). Números de referência desta execução no servidor (mesma semente): SVM-RBF F1 CV **0,820**; GB 0,803; RF 0,798. Teste 25%: acurácia 81,3% (SVM).

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

feat = [c for c in df.columns if c not in ("id", "split", "label")]
X, y = df[feat].values, df.label.values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
modelos = {"RandomForest": RandomForestClassifier(300, random_state=42, n_jobs=-1),
           "GradientBoosting": GradientBoostingClassifier(random_state=42),
           "SVM-RBF": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale"))}
res = []
for nome, m in modelos.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="f1").mean()
    m.fit(Xtr, ytr); pred = m.predict(Xte)
    score = m.predict_proba(Xte)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(Xte)
    res.append({"modelo": nome, "f1_cv": round(s, 3), "acuracia_teste": round(accuracy_score(yte, pred), 3),
                "auc_teste": round(roc_auc_score(yte, score), 3)})
    print(nome, res[-1]); print(classification_report(yte, pred, target_names=["fake", "real"]))
pd.DataFrame(res)

## Sensibilidade a subconjuntos de features (RF, CV 5×)
MFCC=0,750 · Espectrais=0,682 · Chroma=0,648 · Bandas=0,585 · Tudo=**0,801** — MFCC carrega o sinal; bandas+flatness dão ganho incremental.

In [ ]:
grupos = {"MFCC": [c for c in feat if c.startswith("mfcc")],
          "Espectrais": [c for c in feat if c.split('_')[0] in ('centroid','bandwidth','rolloff','flatness','rms','zcr')],
          "Bandas": [c for c in feat if c.startswith('band')],
          "Chroma": [c for c in feat if c.startswith('chroma')], "Tudo": feat}
rf = RandomForestClassifier(200, random_state=42, n_jobs=-1)
sens = {g: round(cross_val_score(rf, df[c].values, y, cv=cv, scoring='f1').mean(), 3) for g, c in grupos.items()}
pd.Series(sens).sort_values().plot.barh(title="F1 (CV) por subconjunto de features"); plt.show()
sens

## Conclusões
- SVM-RBF com features clássicas atinge ~81% de acurácia na detecção binária de ataque no LA — coerente com a literatura para pipelines MFCC; estado da arte (~4% EER) exige modelos espectrais profundos (AASIST).
- A banda 4–8 kHz e a *flatness* confirmam a hipótese de artefatos de síntese no teto espectral.
- Limitações: só condições LA; amostra de 1.200; dataset já normalizado por terceiros (pode suprimir artefatos de amplitude).